# Exploração multimodal — Monitoramento de Pacientes (FIAP TC Fase 4)

Notebook de prototipagem: gera dados sintéticos, executa os detectores de
anomalia e o pipeline multimodal, e visualiza os achados.

> Execute a partir da raiz do projeto (o `sys.path` aponta para `src/`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from multimodal_monitor.synthetic import generate_vitals
from multimodal_monitor.anomaly_detection.vitals import VitalsAnomalyDetector

df = generate_vitals(inject_anomalies=True)
df.head()

In [ ]:
findings = VitalsAnomalyDetector().detect(df)
print(f'{len(findings)} achados detectados')
for f in findings[:10]:
    print(f'- [{f.severity.value}] {f.description}')

In [ ]:
# Visualização da SpO2 (dessaturação plantada)
import matplotlib.pyplot as plt  # opcional: pip install matplotlib
plt.figure(figsize=(10, 3))
plt.plot(df['timestamp'], df['spo2'])
plt.axhline(88, color='red', ls='--', label='limite crítico')
plt.title('SpO2 ao longo do tempo'); plt.legend(); plt.tight_layout()

In [ ]:
# Pipeline multimodal completo
from multimodal_monitor.integration.orchestrator import MonitoringInput, PatientMonitor
from multimodal_monitor.synthetic import generate_prescriptions, generate_pose_frames

data = MonitoringInput(
    patient_id='PAC-NB',
    vitals=df,
    prescriptions=generate_prescriptions(),
    pose_frames=generate_pose_frames(),
)
report = PatientMonitor(dispatch_alerts=False).run(data)
print('Score de risco:', report.risk_score)
print('Alerta:', report.alert.title if report.alert else 'nenhum')